<a href="https://colab.research.google.com/github/mf2056/Dissertation/blob/main/Dataset_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
# Installing all necessary libraries
!pip install pystac-client rasterio rioxarray matplotlib seaborn s3fs
!pip install --extra-index-url https://artifactory.vgt.vito.be/api/pypi/python-packages/simple terracatalogueclient

import os
import zipfile
from google.colab import userdata
from pystac_client import Client
import requests
import numpy as np
import rasterio
import matplotlib.pyplot as plt
import seaborn as sns
from rasterio.enums import Resampling
from terracatalogueclient import Catalogue
from shapely.geometry import Polygon
import rioxarray
import glob
from skimage.util import view_as_windows
from sklearn.model_selection import train_test_split
import s3fs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.5/202.5 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 99.9 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: botocore
    Found existing installation: botocore 1.42.67
    Uninstalling botocore-1.42.67:
      Successfully uninstalled botocore-1.42.67
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
boto3 1.42.67 requires botocore<1.43.0,>=1.42.67, but you have botocore 1.42.61 which is incompatible.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.2.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but

Looking in indexes: https://pypi.org/simple, https://artifactory.vgt.vito.be/api/pypi/python-packages/simple
  Using cached botocore-1.42.67-py3-none-any.whl.metadata (5.9 kB)
Using cached botocore-1.42.67-py3-none-any.whl (14.7 MB)
  Attempting uninstall: botocore
    Found existing installation: botocore 1.42.61
    Uninstalling botocore-1.42.61:
      Successfully uninstalled botocore-1.42.61
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
aiobotocore 3.2.1 requires botocore<1.42.62,>=1.42.53, but you have botocore 1.42.67 which is incompatible.


In [18]:
# Required Details
cop_user = userdata.get('COP_USER')
terra_user = userdata.get('TERRA_USER')
password = userdata.get('PASS')

# Connecting to CDSE STAC API
api_url = "https://stac.dataspace.copernicus.eu/v1"
catalog = Client.open(api_url)

# Searching for data
search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=[54.0, 24.0, 55.0, 25.0],
    datetime="2021-11-01/2022-02-28",
    query={"eo:cloud_cover": {"lt": 5}}
)
items = list(search.items())
print(f"Found {len(items)} products.")

# Downloading selected bands
os.makedirs('sentinel_data', exist_ok=True)
item = items[0] # Taking the first product

band_map = {
    'B02': 'B02_10m', 'B03': 'B03_10m', 'B04': 'B04_10m', 'B08': 'B08_10m',
    'B05': 'B05_20m', 'B06': 'B06_20m', 'B07': 'B07_20m', 'B11': 'B11_20m', 'B12': 'B12_20m'
}

# 1. Initialize S3 access (only need to do this once)
fs = s3fs.S3FileSystem(
    key=cop_user,
    secret=password,
    client_kwargs={'endpoint_url': 'https://s3.dataspace.copernicus.eu'}
)

# 3. Iterate through your band_map and download
for band_name, asset_key in band_map.items():
    # Remove the 's3://' prefix so s3fs recognizes the file path correctly
    s3_path = item.assets[asset_key].href.replace("s3://", "")

    print(f"Downloading {band_name} via S3...")
    try:
        # Transfer file directly from cloud storage to your Colab instance
        fs.get(s3_path, f"sentinel_data/{band_name}.jp2")
        print(f"  Successfully saved {band_name}.jp2")
    except Exception as e:
        print(f"  Failed to download {band_name}: {e}")

print("Download complete.")

Found 130 products.


InvalidSchema: No connection adapters were found for 's3://eodata/Sentinel-2/MSI/L2A_N0500/2022/02/25/S2B_MSIL2A_20220225T065839_N0510_R063_T40QBM_20240530T024034.SAFE/GRANULE/L2A_T40QBM_A025972_20220225T070655/IMG_DATA/R10m/T40QBM_20220225T065839_B02_10m.jp2'

In [ ]:
# Downloading ESA WorldCover
catalogue = Catalogue().authenticate_non_interactive(terra_user, password)
geometry = Polygon.from_bounds(54.0, 24.0, 55.0, 25.0)

# Using 2021 WorldCover Dataset as it is the most recent validated data
wc_products = catalogue.get_products("urn:eop:VITO:ESA_WorldCover_10m_2021_V2", geometry=geometry)

# Downloading to a specific folder in Colab session
if not os.path.exists('worldcover_labels'):
    os.makedirs('worldcover_labels')
catalogue.download_products(list(wc_products), "worldcover_labels")

In [ ]:
def create_aligned_dataset(base_path, label_path, output_stack_path):
    bands = ['B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B11', 'B12']
    band_paths = []

    # Finding band paths
    for band in bands:
        # Looking for files like *B02*.jp2, *B03*.jp2, etc.
        pattern = os.path.join(base_path, f"*{band}*.jp2")
        matches = glob.glob(pattern)
        if matches:
            # Append the first match found for this band
            band_paths.append(matches[0])

    # Ensuring we found all 9 bands
    if len(band_paths) < len(bands):
        raise FileNotFoundError(f"Expected {len(bands)} bands, but only found {len(band_paths)}. Check base_path.")

    # Stacking and Upsampling bands to 10m
    with rasterio.open(band_paths[0]) as src:
        meta = src.meta
        target_h, target_w = src.height, src.width

    meta.update(count=len(bands), driver='GTiff')
    with rasterio.open(output_stack_path, 'w', **meta) as dst:
        for i, path in enumerate(sorted(band_paths), start=1):
            with rasterio.open(path) as src:
                # Upsampling/Resampling
                data = src.read(1, out_shape=(target_h, target_w), resampling=Resampling.bilinear)
                dst.write_band(i, data)

    # Aligning ESA WorldCover to the Sentinel-2 Stack
    s2_rio = rioxarray.open_rasterio(output_stack_path)
    label_rio = rioxarray.open_rasterio(label_path)

    # Matching projection and resolution
    aligned_labels = label_rio.rio.reproject_match(s2_rio)
    aligned_labels.rio.to_raster("aligned_worldcover.tif")
    print(f"Alignment Complete: '{output_stack_path}' and 'aligned_worldcover.tif' are ready.")

In [ ]:
# Accessing the directory where images were downloaded
sentinel_image_dir = "sentinel_data"

# Finding .tif file in Worldcover folder
label_files = glob.glob("worldcover_labels/*.tif", recursive=True)
if label_files:
    target_label = label_files[0]
    print(f"Target Label File: {target_label}")
else:
    print("Label file not found")

# Running alignment function using the detected paths
create_aligned_dataset(sentinel_image_dir, target_label, "9band_stack.tif")

In [ ]:
def perform_eda(stack_path, label_path):
    with rasterio.open(stack_path) as src:
        # Load RGB for visualization (B4, B3, B2 are layers 3, 2, 1 in our stack)
        rgb = src.read([3, 2, 1]).transpose(1, 2, 0)
        rgb = np.clip(rgb / 3000, 0, 1) # Simple normalization for viewing

    with rasterio.open(label_path) as lbl:
        labels = lbl.read(1)

    # Visualization
    fig, ax = plt.subplots(1, 2, figsize=(15, 7))
    ax[0].imshow(rgb)
    ax[0].set_title("Sentinel-2 RGB (Abu Dhabi)")
    ax[1].imshow(labels, cmap='terrain')
    ax[1].set_title("Aligned ESA WorldCover Labels")
    plt.show()

    # Class Distribution Analysis
    unique, counts = np.unique(labels, return_counts=True)
    class_dict = {
        10: "Trees", 20: "Shrubland", 30: "Grassland", 40: "Cropland",
        50: "Built-up", 60: "Bare/Sparse", 70: "Snow/Ice", 80: "Water",
        90: "Herbaceous Wetland", 95: "Mangroves"
    }

    print("--- Class Distribution ---")
    for u, c in zip(unique, counts):
        name = class_dict.get(u, "Unknown")
        percentage = (c / labels.size) * 100
        print(f"{name} ({u}): {c} pixels ({percentage:.2f}%)")

perform_eda("9band_stack.tif", "aligned_worldcover.tif")

In [ ]:
# Patch Extraction for the Models

def extract_patches(image_path, label_path, patch_size, step):
    # Loading Data
    with rasterio.open(image_path) as src:
        img = src.read()                     # Reading all 9 bands, shape becomes (9, Height, Width)
        img = np.moveaxis(img, 0, -1)        # Moving bands to the last dimension for Deep Learning (Height, Width, 9)

    with rasterio.open(label_path) as src:
        label = src.read(1)          # (Height, Width)

    # Extracting patches using sliding windows
    # Note: view_as_windows expects the patch shape as (Height, Width, Bands)
    img_patches = view_as_windows(img, (patch_size, patch_size, 9), step=step)
    label_patches = view_as_windows(label, (patch_size, patch_size), step=step)

    # Reshaping into a flat list
    # view_as_windows output shape is (patches_y, patches_x, 1, patch_size, patch_size, channels)
    # We reshape to: (Total_Patches, Patch_Height, Patch_Width, Channels)
    img_patches = img_patches.reshape(-1, patch_size, patch_size, 9)
    label_patches = label_patches.reshape(-1, patch_size, patch_size)

    return img_patches, label_patches

# Create 256x256 patches for U-Net (X being the raw images and Y being the Labels)
X_unet, y_unet = extract_patches("9band_stack.tif", "aligned_worldcover.tif", 256, 256)

# Create 32x32 patches for Hybrid Model
X_hybrid, y_hybrid = extract_patches("9band_stack.tif", "aligned_worldcover.tif", 32, 32)

print(f"U-Net Dataset: {X_unet.shape[0]} patches of size 256x256x9")
print(f"Hybrid Dataset: {X_hybrid.shape[0]} patches of size 32x32x9")

In [ ]:
# Test-Train Split 80-20

X_train_hybrid, X_test_hybrid, y_train_hybrid, y_test_hybrid = train_test_split(
    X_hybrid, y_hybrid, test_size=0.2, random_state=20, shuffle=True
)

print(f"Training set: {X_train_hybrid.shape[0]} patches")
print(f"Testing set: {X_test_hybrid.shape[0]} patches")

X_train_unet, X_test_unet, y_train_unet, y_test_unet = train_test_split(
    X_unet, y_unet, test_size=0.2, random_state=20, shuffle=True
)

print(f"Training set: {X_train_unet.shape[0]} patches")
print(f"Testing set: {X_test_unet.shape[0]} patches")

In [ ]:
# Normalizing Sentinel-2 raw values from 0-10,000 range to 0-1 for DL model
X_train_un_norm = X_train_unet.astype('float32') / 10000.0
X_test_un_norm = X_test_unet.astype('float32') / 10000.0

# Clip to ensure no values exceed 1.0 due to sensor noise
X_train_un_norm = np.clip(X_train_un_norm, 0, 1)
X_test_un_norm = np.clip(X_test_un_norm, 0, 1)